In [0]:
from pyspark.sql import functions as F

PATH = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABLE = "voebem.bronze.vra"

In [0]:
raw = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(PATH)
)

print("Columns read from file: ")
for col in raw.columns:
    print(f" {col!r}")

In [0]:
RENAME = {
 "ICAO Empresa Aérea": "icao_empresa",
 "Número Voo":"numero_voo",
 "Código Autorização (DI)":"codigo_di",
 "Código Tipo Linha":"codigo_tipo_linha",
 "ICAO Aeródromo Origem":"icao_origem",
 "ICAO Aeródromo Destino":"icao_destino",
 "Partida Prevista":"partida_prevista",
 "Partida Real":"partida_real",
 "Chegada Prevista":"chegada_prevista",
 "Chegada Real":"chegada_real",
 "Situação Voo":"situacao_voo",
 "Código Justificativa":"codigo_justificativa",
}

missing = [col for col in RENAME if col not in raw.columns]
assert not missing, f"Missing columns: {missing} not found in file"

renamed = raw.select(
    *[F.col(f"`{origin}`").cast("string").alias(new) for origin,
    new in RENAME.items()]
)

In [0]:
bronze_vra = (
    renamed
    .withColumn("_arquivo_origem", F.col("_metadata.file_name"))
    .withColumn("_ingerido_em", F.current_timestamp())
)

In [0]:
(
    bronze_vra.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TABLE)
)

print(f"{TABLE}: {spark.table(TABLE).count():,} rows")

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {TABELA} IS
    'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 a jul/2026).
     Dado bruto: todas as colunas string, nenhuma linha descartada.
     Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra/.'
""")